# Tracing

A trace is a timeline of one run. Each timed piece is called a span.

In this notebook, we will:

- create one span;
- connect nested spans into a tree;
- trace an agent and its tools;
- see the data sent through OTLP; and
- send spans to Phoenix when it is running.

In [ ]:
import json
import os
import time
import uuid
from contextlib import contextmanager
from dataclasses import dataclass, replace
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. One span

A span needs a name, a start time, an end time, and an id. A parent id connects it to the span around it. Attributes hold useful details.

In [ ]:
@dataclass
class LearningSpan:
    name: str
    kind: str
    start: float
    end: float | None
    attributes: dict
    span_id: str
    parent_id: str | None

    @property
    def duration_ms(self):
        if self.end is None:
            return None
        return round((self.end - self.start) * 1000, 3)

In [ ]:
span = LearningSpan(
    name="load order",
    kind="tool",
    start=time.time(),
    end=None,
    attributes={"order_id": "ORD-7301"},
    span_id=uuid.uuid4().hex[:16],
    parent_id=None,
)

time.sleep(0.02)
span.end = time.time()

print(span.name, span.duration_ms, "ms")

## 2. A stack creates the tree

The last open span is the parent of the next span. A small stack remembers which spans are open.

In [ ]:
class LearningTracer:
    def __init__(self):
        self.spans = []
        self.open_spans = []

    @contextmanager
    def span(self, name, kind="work", **attributes):
        parent = self.open_spans[-1] if self.open_spans else None
        current = LearningSpan(
            name=name,
            kind=kind,
            start=time.time(),
            end=None,
            attributes=attributes,
            span_id=uuid.uuid4().hex[:16],
            parent_id=parent.span_id if parent else None,
        )
        self.spans.append(current)
        self.open_spans.append(current)
        try:
            yield current
        finally:
            current.end = time.time()
            self.open_spans.pop()

    def show(self):
        depths = {}
        for item in self.spans:
            depth = depths.get(item.parent_id, -1) + 1
            depths[item.span_id] = depth
            print("  " * depth + f"{item.name}: {item.duration_ms} ms")

In [ ]:
demo_trace = LearningTracer()

with demo_trace.span("triage ticket", kind="agent"):
    with demo_trace.span("load order", kind="tool"):
        time.sleep(0.01)
    with demo_trace.span("search policy", kind="tool"):
        time.sleep(0.02)

demo_trace.show()

## 3. Trace a real agent without changing the loop

We can wrap the call site and each tool function. The agent loop itself stays unchanged.

In [ ]:
from shoplab.loop import run_agent
from shoplab.tools import standard_tools
from shoplab.world import load_tickets

agent_trace = LearningTracer()

def add_tool_spans(tools, tracer):
    wrapped = {}

    for name, tool in tools.items():
        original = tool.fn

        def traced_function(_original=original, _name=name, **arguments):
            with tracer.span(_name, kind="tool"):
                return _original(**arguments)

        wrapped[name] = replace(tool, fn=traced_function)

    return wrapped

ticket = load_tickets()["train"][0]
task = json.dumps({key: value for key, value in ticket.items() if key != "gold"})
system = (
    "Handle this Larkspur return. Look up the order, customer, and policy. "
    "Then call finish with decision, policy_id, and refund_usd."
)

with agent_trace.span("triage", kind="agent", ticket_id=ticket["ticket_id"]):
    result = run_agent(
        task,
        add_tool_spans(standard_tools(), agent_trace),
        model=MODEL,
        system=system,
        max_steps=8,
    )

agent_trace.show()
print("answer:", result.answer)

## 4. OTLP is the export shape

Tracing tools store spans in their own objects. OTLP gives them a shared format for sending those spans to a tracing server.

In [ ]:
from shoplab.trace import to_otlp

payload = to_otlp(agent_trace.spans, project="simplified-course")
sent_spans = payload["resourceSpans"][0]["scopeSpans"][0]["spans"]

print("spans in payload:", len(sent_spans))
print(json.dumps(sent_spans[0], indent=2)[:800])

## 5. Phoenix displays the trace

The next cell starts Phoenix if the optional package is available, then exports our hand-built spans.

Phoenix can automatically trace LiteLLM calls. Plain Python tool functions still need manual spans if we want to see them.

In [ ]:
try:
    import obs
    from shoplab.trace import export_phoenix

    obs.enable_phoenix()
    print("export status:", export_phoenix(agent_trace))
except Exception as error:
    print("Phoenix was skipped:", error)

The course provides the finished Span and Tracer classes in shoplab.trace. They use the same ideas: timed spans, parent ids, and a stack of open work.

In [ ]:
from shoplab.trace import Tracer

course_trace = Tracer()
with course_trace.span("outer"):
    with course_trace.span("inner"):
        time.sleep(0.01)

course_trace.print_tree()

## What to remember

1. A span is one timed piece of work.
2. A parent id turns spans into a tree.
3. A stack remembers the current parent.
4. Wrappers can trace an agent without changing its loop.
5. OTLP carries spans to tools such as Phoenix.
6. Automatic tracing only sees the libraries it instruments.

## Small practice steps

1. Add a ticket id attribute to a span.
2. Create three levels of nested spans.
3. Find the matching parentSpanId in the OTLP payload.